In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
import seaborn as sns
import math
from tqdm import tqdm
import os
import re

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import scipy
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, ElasticNetCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import precision_score, recall_score, f1_score

from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test, logrank_test
import warnings
from functools import reduce

import liana as li
import liana.plotting as pl
from liana.method import (
    cellphonedb, connectome, logfc,
    natmi, singlecellsignalr, geometric_mean, genericMistyData
)

from scipy.stats import ttest_ind, fisher_exact
from statsmodels.stats.multitest import multipletests

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

import decoupler as dc
#import mygene

import pickle
import random

In [ ]:
random.seed(42)

In [ ]:
model_name = "v31_norecurr_bulkhvg_MOFA"

# Pathways (sc + spatial) + groups + genes spatial

In [ ]:
pathway_results = pd.read_csv(f"mofa/mofa_workflow/results_mosaic/06_results/06_Pathway_Enrichment_{model_name}.csv")

In [ ]:
pathway_results

In [ ]:
sign = pathway_results[pathway_results["padj"]<=0.05]

In [ ]:
def plot_pathways(data: pd.DataFrame, factor: str):
    sign = data
    fac = sign[sign["variable"]==factor]
    df = fac
    
    df['logpadj'] = -np.log10(df['padj'])
    
    
    views = sorted(df['view'].unique())
    enrichments = sorted(df['enrichment'].unique())
    
    # Determine grid size for subplots
    n_rows = len(views)
    n_cols = len(enrichments)

    print(f"{factor}: Views: {n_rows}, Enrichments: {n_cols}")
    
    # Create subplots with shared x-axis 
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 6), squeeze=False)
    
    # Loop through each combination of view and enrichment
    for i, view in enumerate(views):
        for j, enrich in enumerate(enrichments):
            ax = axes[i, j]
            # Subset the dataframe for the current view and enrichment
            subset = df[(df['view'] == view) & (df['enrichment'] == enrich)]
            
            # Only plot if the subset is not empty
            if not subset.empty:
                # Sort by -log10(padj) so that the most significant (largest value) is at the top
                subset = subset.sort_values('logpadj', ascending=False)
                ax.barh(subset['pathway_name'], subset['logpadj'])
                ax.invert_yaxis()  # Invert y-axis to have the highest value on top
            
            # Add a title and labels for clarity
            ax.set_title(f"View: {view} | Enrichment: {enrich}")
            if i == n_rows - 1:
                ax.set_xlabel("-log(padj)")
            if j == 0:
                ax.set_ylabel("Pathway")
    
    plt.tight_layout()
    plt.savefig(f"mofa/resources/{model_name}_{factor}.pdf", bbox_inches="tight")
    #plt.show()
    

In [ ]:
for factor in sign["variable"].unique():
    plot_pathways(sign, factor)

In [ ]:
# spatial distribution of pathways:

In [ ]:
adata = sc.read_h5ad("mofa/resources/v2_visium_concat_deconvolved_pathways_patho_hoptimus1.h5ad")

In [ ]:
adata

In [ ]:
patho_sample_ids = ['HKG052a', 'HKG021a', 'HKG001a', 'HKG028a', 'HKG002a', 'HKG011a', 'HKG015a']

In [ ]:
merged_meta_mofa = pd.read_csv(f"mosaic_samples_pfs_os_cluster_{model_name}.csv", index_col=0)
merged_meta_mofa

In [ ]:
short = merged_meta_mofa[merged_meta_mofa["os_years"]<1.5]["sample_id"].values
medium = merged_meta_mofa[(merged_meta_mofa["os_years"]>1.5) & (merged_meta_mofa["os_years"]<3)]["sample_id"].values
long = merged_meta_mofa[merged_meta_mofa["os_years"]>3]["sample_id"].values

In [ ]:
vis_samples = list(adata.obs["sample_id"].unique())

In [ ]:
long = [sample for sample in long if sample in vis_samples]
long

In [ ]:
medium # 40, 110 not visium, 38,65
medium = [sample for sample in medium if sample in vis_samples]
medium

In [ ]:
short # 75, 77, 108, 76, 74 not in visium
short = [sample for sample in short if sample in vis_samples]
short

In [ ]:
indices_long = np.linspace(
    start=0,
    stop=len(long) - 1,
    num=5
).astype(int)

indices_medium = np.linspace(
    start=0,
    stop=len(medium) - 1,
    num=5
).astype(int)

indices_short = np.linspace(
    start=0,
    stop=len(short) - 1,
    num=5
).astype(int)


In [ ]:
for i in indices_long:
    sample_id = long[i]
    sample = adata[adata.obs['sample_id']==sample_id]
    #sc.pl.spatial(sample, color=['INTERFERON_ALPHA_RESPONSE', 'INTERFERON_GAMMA_RESPONSE', 'HYPOXIA', 'APICAL_JUNCTION', 'OXIDATIVE_PHOSPHORYLATION', 'MYOGENESIS', 'COAGULATION','E2F_TARGETS' ], spot_size=150, save=f"_long_{model_name}_{sample_id}.pdf")
    #sc.pl.spatial(sample, color=['AC-like', 'Astrocyte', 'B cell', 'CD4/CD8', 'DC', 'E-MDSCs', 'Endothelial', 'M-MDSCs', 'MES-like', 'Mast', 'Mono', 'Mural cell', 'NK', 'NPC-like', 'Neuron', 'Neutrophil', 'OPC', 'OPC-like', 'Oligodendrocyte', 'Plasma B', 'RG', 'TAM-BDM', 'TAM-MG', 'TRAIL+ Astrocytes'], spot_size=150, save=f"_long_ctabundance_{model_name}_{sample_id}.pdf")
    sc.pl.spatial(sample, color=['TFCP2L1','HMGA2', 'SLC16A1', 'SLC16A3', 'ENTPD2', 'F2RL2', 'F3', 'LRRC15', 'COX14', 'TCF21', 'MXRA5'], spot_size=150, save=f"_genes_long_{model_name}_{sample_id}.pdf")


In [ ]:
for i in indices_medium:
    sample_id = medium[i]
    sample = adata[adata.obs['sample_id']==sample_id]
    #sc.pl.spatial(sample, color=['INTERFERON_ALPHA_RESPONSE', 'INTERFERON_GAMMA_RESPONSE', 'HYPOXIA', 'APICAL_JUNCTION', 'OXIDATIVE_PHOSPHORYLATION', 'MYOGENESIS', 'COAGULATION','E2F_TARGETS' ], spot_size=150, save=f"_medium_{model_name}_{sample_id}.pdf")
    #sc.pl.spatial(sample, color=['AC-like', 'Astrocyte', 'B cell', 'CD4/CD8', 'DC', 'E-MDSCs', 'Endothelial', 'M-MDSCs', 'MES-like', 'Mast', 'Mono', 'Mural cell', 'NK', 'NPC-like', 'Neuron', 'Neutrophil', 'OPC', 'OPC-like', 'Oligodendrocyte', 'Plasma B', 'RG', 'TAM-BDM', 'TAM-MG', 'TRAIL+ Astrocytes'], spot_size=150, save=f"_medium_ctabundance_{model_name}_{sample_id}.pdf")
    sc.pl.spatial(sample, color=['TFCP2L1','HMGA2', 'SLC16A1', 'SLC16A3', 'ENTPD2', 'F2RL2', 'F3', 'LRRC15', 'COX14', 'TCF21','MXRA5'], spot_size=150, save=f"_genes_medium_{model_name}_{sample_id}.pdf")


In [ ]:
for i in indices_short:
    sample_id = short[i]
    sample = adata[adata.obs['sample_id']==sample_id]
    #sc.pl.spatial(sample, color=['INTERFERON_ALPHA_RESPONSE', 'INTERFERON_GAMMA_RESPONSE', 'HYPOXIA', 'APICAL_JUNCTION', 'OXIDATIVE_PHOSPHORYLATION', 'MYOGENESIS', 'COAGULATION','E2F_TARGETS' ], spot_size=150, save=f"_short_{model_name}_{sample_id}.pdf")
    #sc.pl.spatial(sample, color=['AC-like', 'Astrocyte', 'B cell', 'CD4/CD8', 'DC', 'E-MDSCs', 'Endothelial', 'M-MDSCs', 'MES-like', 'Mast', 'Mono', 'Mural cell', 'NK', 'NPC-like', 'Neuron', 'Neutrophil', 'OPC', 'OPC-like', 'Oligodendrocyte', 'Plasma B', 'RG', 'TAM-BDM', 'TAM-MG', 'TRAIL+ Astrocytes'], spot_size=150, save=f"_short_ctabundance_{model_name}_{sample_id}.pdf")
    sc.pl.spatial(sample, color=['TFCP2L1','HMGA2', 'SLC16A1', 'SLC16A3', 'ENTPD2', 'F2RL2', 'F3', 'LRRC15', 'COX14', 'TCF21', 'MXRA5'], spot_size=150, save=f"_genes_short_{model_name}_{sample_id}.pdf")


# patient stratification

In [ ]:
COLOR_PALETTE = ['green', 'red','#1f77b4','#ff7f0e','#2ca02c', '#d62728', '#9467bd',
                 '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

In [ ]:
def mofa_patient_stratification(
    csv_path: str,
    include_factors: list[str] | None = None,
    exclude_factors: list[str] | None = None,
    k_elbow: range = range(1, 11),
    k_sil: range = range(2, 11),
    random_state: int = 42
) -> pd.DataFrame:
    """
    1. Loads MOFA factors from CSV (expects Factor1–FactorN + sample_id).
    2. Optionally includes/excludes specified factors.
    3. Plots elbow curve (inertia vs k) for k in k_elbow.
    4. Plots silhouette scores for k in k_sil and prints best k.
    5. Clusters with the best silhouette k, runs PCA→2D, and shows a scatter.
    6. Returns the DataFrame with an added 'Cluster' column.
    """
    # Load and preprocess 
    df = (
        pd.read_csv(csv_path, index_col=0)
          .reset_index()
          .rename(columns={'index': 'Factor1'})
          .set_index('sample_id')
    )
    factor_cols = [c for c in df.columns if c.startswith('Factor')]

    # Ablation: include/exclude factors
    if include_factors is not None:
        factor_cols = [f for f in include_factors if f in factor_cols]
    if exclude_factors is not None:
        factor_cols = [f for f in factor_cols if f not in exclude_factors]

    df_factors = df[factor_cols].copy()
    print(f"Using factors: {factor_cols}\n")


    # Case 1: Only one factor is provided. Split by the median.
    if len(factor_cols) == 1:
        print("Only one factor provided. Splitting by median into 'low' (0) and 'high' (1) groups.\n")
        single_factor = factor_cols[0]

        # Median split logic
        median_value = df_factors[single_factor].median()
        print(f"Median factor value used for split: {median_value}")
        # Assigns 1 to 'high' group (> median), 0 to 'low' group (<= median)
        df['Cluster'] = (df_factors[single_factor] > median_value).astype(int)
        best_k = 2 

        # Prepare for visualization 
        # Plot the single factor on the x-axis and use a constant y-axis
        coords = np.zeros((len(df_factors), 2))
        coords[:, 0] = df_factors[single_factor].values
        xlabel, ylabel = single_factor, '' 

    # Case 2: More than one factor. Use the K-Means approach.
    else:
        print("Multiple factors provided. Using K-Means for clustering.\n")

        # Elbow method 
        inertias = []
        for k in k_elbow:
            km = KMeans(n_clusters=k, random_state=random_state, n_init='auto').fit(df_factors)
            inertias.append(km.inertia_)
        plt.figure(figsize=(6,4)); plt.plot(list(k_elbow), inertias, 'o-'); plt.xlabel('Number of clusters k')
        plt.ylabel('Inertia'); plt.title('Elbow Method for Optimal k'); plt.xticks(list(k_elbow)); plt.tight_layout(); plt.show()

        # Silhouette analysis 
        sil_scores = []
        for k in k_sil:
            labels = KMeans(n_clusters=k, random_state=random_state, n_init='auto').fit_predict(df_factors)
            sil_scores.append(silhouette_score(df_factors, labels))
        plt.figure(figsize=(6,4)); plt.plot(list(k_sil), sil_scores, 'o-'); plt.xlabel('Number of clusters k')
        plt.ylabel('Silhouette Score'); plt.title('Silhouette Analysis for Optimal k'); plt.xticks(list(k_sil)); plt.tight_layout(); plt.show()

        best_k = k_sil[sil_scores.index(max(sil_scores))]
        print(f"Best k by silhouette: {best_k}\n")
        best_k = 2
        
        # Fit KMeans and prepare for visualization
        df['Cluster'] = KMeans(n_clusters=best_k, random_state=random_state, n_init='auto').fit_predict(df_factors)

        if df_factors.shape[1] == 2:
            print("Input is 2D, plotting raw factor values.\n")
            coords = df_factors.values; xlabel, ylabel = df_factors.columns[0], df_factors.columns[1]
        else:
            print("Input is >2D, using PCA for visualization.\n")
            pca = PCA(n_components=2, random_state=random_state)
            coords = pca.fit_transform(df_factors); xlabel, ylabel = 'PC1', 'PC2'

    plt.figure(figsize=(6, 5))


    # Manually map cluster IDs to the color palette for direct control
    point_colors = df['Cluster'].map(lambda cid: COLOR_PALETTE[cid % len(COLOR_PALETTE)])

    plt.scatter(
        coords[:, 0], coords[:, 1],
        c=point_colors,  # Pass the explicit list of colors
        s=50, alpha=0.8
    )
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(f'Patient Stratification (k={best_k})')

    # Manually create legend handles to ensure they match the palette
    unique_clusters = sorted(df['Cluster'].unique())
    legend_handles = [
        mpatches.Patch(color=COLOR_PALETTE[i % len(COLOR_PALETTE)], label=f'Cluster {i}')
        for i in unique_clusters
    ]
    plt.legend(handles=legend_handles)

    plt.tight_layout()
    plt.savefig(f"figures/{model_name}_scatter.pdf")
    #plt.show()

    return df

In [ ]:
def merge_clinical_data(
    result_df: pd.DataFrame,
    meta_path: str,
    clinical_cols: list[str] = ['os_years','os_censor','pfs_years','pfs_censor'],
    id_col: str = 'sample_id'
) -> pd.DataFrame:
    """
    Reads the clinical CSV, cleans up sample IDs, and merges with result_df.
    """
    meta = pd.read_csv(meta_path)
    # normalize sample_id format (remove underscores)
    meta[id_col] = meta[id_col].str.replace('_', '')
    meta = meta.set_index(id_col)
    
    # join on the requested clinical columns
    merged = result_df.join(meta[clinical_cols], how='inner')
    return merged

In [ ]:
# with p val plot
def plot_km_and_logrank(
    df: pd.DataFrame,
    metric: str = 'pfs',      # either 'os' or 'pfs'
    cluster_col: str = 'Cluster',
    save: bool = True
) -> None:
    """
    Plots Kaplan–Meier curves and prints overall log-rank p-value for the given metric,
    and also annotates the p-value on the plot.
    Expects df to have:
      - f"{metric}_years" 
      - f"{metric}_censor"
      - cluster_col
    """
    time_col  = f'{metric}_years'
    event_col = f'{metric}_censor'
    
    # create figure & axis explicitly
    fig, ax = plt.subplots(figsize=(7,5))
    kmf = KaplanMeierFitter()

    # plot each cluster
    for cid, group in df.groupby(cluster_col):
        kmf.fit(
            durations=group[time_col],
            event_observed=group[event_col],
            label=f'Cluster {cid}'
        )
        kmf.plot(ci_show=True, ax=ax, color=COLOR_PALETTE[cid])

    ax.set_title(f'{metric.upper()} by MOFA Cluster')
    ax.set_xlabel(f'{metric.upper()} (years)')
    ax.set_ylabel('Survival probability')
    ax.legend()
    plt.tight_layout()

    # overall log-rank test
    results = multivariate_logrank_test(
        df[time_col],
        df[cluster_col],
        df[event_col]
    )
    p_val = results.p_value
    rounded_p = round(p_val, 5)
    print(f"{metric.upper()} overall log-rank p-value: {rounded_p:.5f}")

    # annotate rounded p-value on the plot
    ax.text(
        0.18, 0.02,
        f'p = {rounded_p:.5f}',
        ha='right', va='bottom',
        transform=ax.transAxes,
        fontsize=12,
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.8)
    )

    if save:
        fig.savefig(f"figures/os_kaplan_meier_{model_name}.pdf")

In [ ]:
result_df = mofa_patient_stratification(
    #'03_Factor_Data_v14_mosaic_sc_rounded_GBmap_mdsc_visium_lucas_MOFA.csv',
    f'mofa/mofa_workflow/results_mosaic/03_results/03_Factor_Data_{model_name}.csv',
    include_factors=["Factor7"]
)

In [ ]:
df_merged = merge_clinical_data(
     result_df,
     '/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/data/mosaic_dataset/Clinical/GBM_HK_sample_and_clinical_data.csv'
     )

In [ ]:
df_merged.to_csv(f"mofa/resources/patient_strat_{model_name}.csv")

In [ ]:
plot_km_and_logrank(df_merged, metric='pfs', save=True)

In [ ]:
plot_km_and_logrank(df_merged, metric='os', save=True)

In [ ]:
# investigate what makes the groups different

In [ ]:
data1 = pd.read_csv(f"mofa/resources/patient_strat_{model_name}.csv")
groups = data1[['sample_id', 'Cluster']]

In [ ]:
groups

## bulk DEGs

In [ ]:
data2 = pd.read_csv("mofa/mofa_workflow/input_data_mosaic/bulk_05symbols.csv")

In [ ]:
data2.set_index('sample_id', inplace=True)
samples_in_both = data2.index.intersection(groups["sample_id"])
data2 = data2.loc[samples_in_both, :]

meta = (
    groups
    .set_index("sample_id")
    .loc[samples_in_both]  
    .copy()
)

# Ensure Cluster is categorical strings
meta = meta.copy()
meta["Cluster"] = meta["Cluster"].astype(str).astype("category")


# Transpose -> samples x genes for PyDESeq2
counts_t = data2

# Run DE 
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

dds = DeseqDataSet(
    counts=counts_t,
    metadata=meta,
    design_factors="Cluster",
    refit_cooks=True,
)
dds.deseq2()

stats = DeseqStats(dds, contrast=["Cluster", "1", "0"])
stats.summary()
try:
    stats.lfc_shrink()
except Exception:
    pass

res = stats.results_df


# Clean up results, add convenience columns, save table if desired

res["padj_plot"] = res["padj"].fillna(1.0).replace([np.inf, -np.inf], 1.0)
res["neglog10_padj"] = -np.log10(res["padj_plot"].clip(lower=np.finfo(float).tiny))

# Add significance flag 
log2fc_col = "log2FoldChange" if "log2FoldChange" in res.columns else "log2FoldChange_shrunken"
if log2fc_col not in res.columns:  # fallback
    log2fc_col = "log2FoldChange"

res["signif"] = (res["padj_plot"] < 0.05) & (res[log2fc_col].abs() > 1)

# write results to disk
res.to_csv(f"mofa/resources/{model_name}deseq2_cluster1_vs_0_results.csv") # SAVED HERE


# Volcano plot

plt.figure(figsize=(8, 6))

# Non-significant
ns = res[~res["signif"]]
plt.scatter(ns[log2fc_col], ns["neglog10_padj"], s=10, alpha=0.6)

# Significant
sg = res[res["signif"]]
plt.scatter(sg[log2fc_col], sg["neglog10_padj"], s=12, alpha=0.9)

# Horizontal line at FDR 0.05
thr = -np.log10(0.05)
plt.axhline(y=thr, linestyle="--", linewidth=1)

# Vertical lines at |log2FC| = 1
plt.axvline(x=1, linestyle="--", linewidth=1)
plt.axvline(x=-1, linestyle="--", linewidth=1)

plt.xlabel("log2 fold change (Cluster 1 vs 0)")
plt.ylabel("-log10 adjusted p-value")
plt.title("Volcano plot: Cluster 1 vs Cluster 0 (DESeq2)")

# Annotate top genes by padj 
top_n = 10
top_hits = (
    res[res["signif"]]
    .sort_values("padj_plot", ascending=True)
    .head(top_n)
)
for gene, row in top_hits.iterrows():
    plt.text(
        row[log2fc_col],
        row["neglog10_padj"],
        str(gene),
        fontsize=8,
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.savefig(f"figures/{model_name}_bulk_voclano_patient_strat.pdf") # SAVED HERE
plt.show()


# Quick summary of hit counts (optional)

n_sig = int(res["signif"].sum())
n_up = int(((res["padj_plot"] < 0.05) & (res[log2fc_col] > 1)).sum())
n_down = int(((res["padj_plot"] < 0.05) & (res[log2fc_col] < -1)).sum())
print(f"Significant (FDR<0.05 & |log2FC|>1): {n_sig} (up: {n_up}, down: {n_down})")

In [ ]:
top_hits = (
    res[res["signif"]]
    .sort_values("padj_plot", ascending=True)
)

In [ ]:
up = top_hits[top_hits["log2FoldChange"]>0].index.values
down = top_hits[top_hits["log2FoldChange"]<0].index.values

In [ ]:
up

In [ ]:
down

## cell type specific DEGs

In [ ]:
adata = sc.read_h5ad("mofa/mofa_workflow/input_data_mosaic/sc_mosaic_rounded_embeds_NOuncert_subclusters_mofa.h5ad")

In [ ]:
adata

In [ ]:
groups

In [ ]:
warnings.filterwarnings('ignore')

sample_col = "sample_id"
celltype_col = "cluster_id"
condition_col = "Cluster"
min_samples_per_group = 3 # Minimum number of samples in EACH condition for a celltype to be tested


# Build per-sample cell-type composition table 

print("Step 1 & 2: Processing cell compositions and clinical metadata...")
if sample_col not in adata.obs.columns:
    raise KeyError(f"'{sample_col}' not found in adata.obs")
if celltype_col not in adata.obs.columns:
    raise KeyError(f"'{celltype_col}' not found in adata.obs")

obs = adata.obs[[sample_col, celltype_col]].copy()
counts = obs.value_counts([sample_col, celltype_col]).unstack(fill_value=0)
props = counts.div(counts.sum(axis=1), axis=0)
props.index.name = sample_col
props.reset_index(inplace=True)


# Load and normalize clinical metadata
clin = groups

if sample_col not in clin.columns:
    raise KeyError(f"'{sample_col}' not found in clinical CSV")
if condition_col not in clin.columns:
    raise KeyError(f"'{condition_col}' not found in clinical CSV")

clin[condition_col] = clin[condition_col].astype(str).str.strip()

clin = clin.dropna(subset=[condition_col])

print("Clinical metadata processed.\n")

# Create Pseudo-bulk AnnData object
print("Step 3: Creating pseudo-bulk expression matrix...")
pb_adata = dc.pp.pseudobulk(
    adata,
    sample_col=sample_col,
    groups_col=celltype_col,
    layer=None, # Use adata.X,
    mode='sum'
)

min_cells_to_keep=10, # Minimum number of cells to create a pseudo-bulk sample
min_counts_to_keep=100 # Minimum number of counts to create a pseudo-bulk sample

# Filter pseudo-bulk samples based on number of cells and total counts
n_samps_before = pb_adata.n_obs
pb_adata = pb_adata[pb_adata.obs['psbulk_cells'] >= min_cells_to_keep].copy()
pb_adata = pb_adata[pb_adata.X.sum(axis=1) >= min_counts_to_keep].copy()
n_samps_after = pb_adata.n_obs
print(f"  > Filtered pseudo-bulk samples. Kept {n_samps_after} out of {n_samps_before}.")

# Merge clinical information into the pseudo-bulk object
pb_adata.obs = pb_adata.obs.reset_index().merge(
    clin[[sample_col, condition_col]], on=sample_col, how='left'
).set_index('index')

# Remove samples that don't have clinical metadata
pb_adata = pb_adata[~pb_adata.obs[condition_col].isna()].copy()

print(f"Pseudo-bulk AnnData created with {pb_adata.n_obs} samples and {pb_adata.n_vars} genes.\n")


# Filter cell types that lack sufficient samples in both conditions

print("Step 4: Filtering cell types for robust DEG analysis...")
# Count samples per cell type and condition
counts_per_group = pb_adata.obs.groupby([celltype_col, condition_col]).size().unstack(fill_value=0)

# Identify cell types that meet the minimum sample requirement in both groups
keep_celltypes = (counts_per_group >= min_samples_per_group).all(axis=1)
celltypes_to_test = keep_celltypes[keep_celltypes].index.tolist()

print(f"Cell types meeting the criteria (>= {min_samples_per_group} samples per group):")
if not celltypes_to_test:
    print("  None. The script will stop.")
else:
    for ct in celltypes_to_test:
        print(f"  - {ct}")

print("\nFull sample breakdown per cell type and condition:")
print(counts_per_group)
print("-" * 50 + "\n")

# Run DEG analysis per cell type

all_deg_results = []

if not celltypes_to_test:
    print("No cell types to test. Exiting.")
else:
    print("Step 5: Running PyDESeq2 for each eligible cell type...")
    for cell_type in celltypes_to_test:
        print(f"  > Processing: {cell_type}")
        # Subset the pseudo-bulk object to the current cell type
        adata_ct = pb_adata[pb_adata.obs[celltype_col] == cell_type].copy()
        
        sc.pp.filter_genes(adata_ct, min_counts=10)
        
        # PyDESeq2 requires integer counts
        adata_ct.X = adata_ct.X.astype(int)
        
        # Build the DESeqDataSet
        dds = DeseqDataSet(
            adata=adata_ct,
            design_factors=condition_col,
            ref_level=[condition_col, "0"] # Set reference for comparison
        )
        
        # Run the DESeq2 analysis
        dds.deseq2()
        
        # Extract results
        stat_res = DeseqStats(dds, contrast=[condition_col, "1", "0"])
        stat_res.summary()
        res_df = stat_res.results_df
        
        # Add metadata and format the results
        res_df['cell_type'] = cell_type
        res_df = res_df.reset_index().rename(columns={'index': 'gene'})
        
        all_deg_results.append(res_df)
        print(f"    ... Done. Found {res_df[res_df['padj'] < 0.05].shape[0]} significant genes (p-adj < 0.05).")



# Consolidate and save results

if all_deg_results:
    print("\nStep 6: Consolidating and saving results...")
    final_results_df = pd.concat(all_deg_results, ignore_index=True)
    
    # Sort results for clarity
    final_results_df.sort_values(by=['cell_type', 'padj'], inplace=True)
    
    output_path = f"mofa/resources/{model_name}_pseudobulk_deg_results_patient_strat.csv"
    final_results_df.to_csv(output_path, index=False)
    
    print(f"\nAnalysis complete! All results saved to '{output_path}'.")
    print(final_results_df.head())
else:
    print("\nAnalysis finished, but no results were generated.")


In [ ]:
res_df={}
for view in final_results_df["cell_type"].unique():
    view_df = final_results_df[final_results_df["cell_type"]==view]
    view_df["signif"] = (view_df["padj"] < 0.05) & (view_df['log2FoldChange'].abs() > 1)
    top_hits = (
        view_df[view_df["signif"]]
        .sort_values("padj", ascending=True)
    )
    up = top_hits[top_hits["log2FoldChange"]>0]['gene'].values
    down = top_hits[top_hits["log2FoldChange"]<0]['gene'].values

    res_df[view] = [up, down]

In [ ]:
res_df

In [ ]:

# Path to the results file from the previous
results_csv_path = output_path
output_dir = "figures"

# Significance thresholds
padj_threshold = 0.05
log2fc_threshold = 1.0 

n_genes_to_label = 10


# Setup

# Check if the results file exists
if not os.path.exists(results_csv_path):
    raise FileNotFoundError(f"Results file not found at '{results_csv_path}'. Please run the main analysis script first.")

# Load the DEG results
try:
    results_df = pd.read_csv(results_csv_path)
except pd.errors.EmptyDataError:
    print("The results file is empty. No plots will be generated.")
    exit()

# Ensure required columns are present
required_cols = ['gene', 'log2FoldChange', 'pvalue', 'padj', 'cell_type']
if not all(col in results_df.columns for col in required_cols):
    raise KeyError(f"The results CSV must contain the following columns: {required_cols}")

print(f"Loaded {len(results_df)} total gene results across {results_df['cell_type'].nunique()} cell types.")


# Set up the Subplot Grid

cell_types = sorted(results_df['cell_type'].unique())
n_plots = len(cell_types)

if n_plots == 0:
    print("No cell types found in the data. Exiting.")
    exit()

# Calculate an aesthetically pleasing grid layout
n_cols = math.ceil(math.sqrt(n_plots))
n_rows = math.ceil(n_plots / n_cols)

# Create the figure and the grid of subplots 
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 4.5), squeeze=False)

# Flatten the 2D array of axes into a 1D array for easy iteration
axes = axes.flatten()

print(f"Creating a {n_rows}x{n_cols} grid for {n_plots} cell types...")


# Generate a plot for each cell type on its subplot

for i, cell_type in enumerate(cell_types):
    ax = axes[i] 

    # Subset the dataframe for the current cell type
    df_ct = results_df[results_df['cell_type'] == cell_type].copy()
    df_ct.dropna(subset=['pvalue', 'log2FoldChange'], inplace=True)

    if df_ct.empty:
        ax.text(0.5, 0.5, 'No significant results', ha='center', va='center')
        ax.set_title(cell_type, fontsize=12, weight='bold')
        ax.set_xticks([])
        ax.set_yticks([])
        continue

    # Create a column for -log10(p-value)
    epsilon = 1e-300
    df_ct['-log10p'] = -np.log10(df_ct['pvalue'] + epsilon)

    # Categorize genes
    def categorize_gene(row):
        is_significant = row['padj'] < padj_threshold
        is_upregulated = row['log2FoldChange'] > log2fc_threshold
        is_downregulated = row['log2FoldChange'] < -log2fc_threshold
        if is_significant and is_upregulated: return 'Upregulated'
        if is_significant and is_downregulated: return 'Downregulated'
        return 'Not significant'
    df_ct['significance'] = df_ct.apply(categorize_gene, axis=1)

    sns.scatterplot(
        data=df_ct,
        x='log2FoldChange',
        y='-log10p',
        hue='significance',
        palette={'Upregulated': '#d62728', 'Downregulated': '#1f77b4', 'Not significant': 'grey'},
        s=15, alpha=0.7, ax=ax, legend=False # Legend will be handled manually
    )

    df_ct_sig = df_ct[df_ct['significance'] != 'Not significant'].sort_values('pvalue')
    top_up = df_ct_sig[df_ct_sig['significance'] == 'Upregulated'].head(n_genes_to_label)
    top_down = df_ct_sig[df_ct_sig['significance'] == 'Downregulated'].head(n_genes_to_label)
    genes_to_label = pd.concat([top_up, top_down])

    for _, row in genes_to_label.iterrows():
        ax.text(row['log2FoldChange'], row['-log10p'], row['gene'], fontsize=7)


    ax.set_title(cell_type, fontsize=12, weight='bold')
    ax.set_xlabel('log2 FC', fontsize=10)
    ax.set_ylabel('-log10(p-value)', fontsize=10)
    ax.axhline(y=-np.log10(0.05), color='black', linestyle='--', linewidth=0.8)
    ax.axvline(x=log2fc_threshold, color='black', linestyle='--', linewidth=0.8)
    ax.axvline(x=-log2fc_threshold, color='black', linestyle='--', linewidth=0.8)



# Finalize and Save the Combined Plot
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)


handles = [plt.Line2D([0], [0], marker='o', color='w', label='Upregulated', markerfacecolor='#d62728', markersize=8),
           plt.Line2D([0], [0], marker='o', color='w', label='Downregulated', markerfacecolor='#1f77b4', markersize=8),
           plt.Line2D([0], [0], marker='o', color='w', label='Not significant', markerfacecolor='grey', markersize=8)]
fig.legend(handles=handles, title='Significance', loc='upper right')


# Adjust layout to prevent titles/labels from overlapping
plt.tight_layout(rect=[0, 0, 0.9, 1]) # Adjust rect to make space for the legend

# Save the combined figure
plt.savefig(f"{output_dir}/{model_name}_volcano_patientstrat.pdf", bbox_inches='tight') # change here if variable changes
plt.close(fig)

print(f"\nCombined volcano plot saved to '{output_dir}/{model_name}_volcano_patientstrat.pdf'.") # change here if variable changes

## cell type abundance differences

In [ ]:
# load cell type proportions

In [ ]:
clr_transformed = pd.read_csv("celltype_proportions_zComp_CLR.csv")
clr_transformed

In [ ]:
groups

In [ ]:
merged = pd.merge(clr_transformed, groups, on="sample_id")

In [ ]:
merged

In [ ]:
cell_type_columns = [col for col in merged.columns if col not in ["sample_id", "Cluster"]]
cell_type_columns

In [ ]:
n_plots = len(cell_type_columns)
ncols = 4
nrows = math.ceil(n_plots / ncols)

custom_palette = {
    "Factor 7 low": "green",
    "Factor 7 high": "red"
}

fig, axes = plt.subplots(nrows, ncols, figsize=(20, 5 * nrows))
axes = axes.flatten()

# Significance helper function
def get_star_label(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'n.s.'

# Renaming mapping for X-axis
cluster_rename = {"0": "Factor 7 low", "1": "Factor 7 high"}
merged_copy = merged.copy()
merged_copy['Cluster_Label'] = merged_copy['Cluster'].astype(str).map(cluster_rename)

for i, cell_type in enumerate(cell_type_columns):
    ax = axes[i]
    
    # Clean cell type name (e.g., TAM.MG -> TAM-MG)
    cell_type_clean = cell_type.replace(".", "-")

    # Stats calculation
    c0 = merged_copy[merged_copy['Cluster'] == 0][cell_type]
    c1 = merged_copy[merged_copy['Cluster'] == 1][cell_type]
    t_stat, p_val = ttest_ind(c0, c1, nan_policy='omit')

    # Plotting (Set2 palette & Black-outlined circles)
    sns.boxplot(x='Cluster_Label', y=cell_type, data=merged_copy, ax=ax, 
                palette=custom_palette, width=0.5, showfliers=False, linewidth=1.2)
    
    sns.stripplot(x='Cluster_Label', y=cell_type, data=merged_copy, ax=ax, 
                  dodge=True, jitter=0.2, size=4, alpha=0.8, 
                  linewidth=0.6, edgecolor='black', palette=custom_palette)

    # Brackets and Stars logic
    y_min, y_max = merged_copy[cell_type].min(), merged_copy[cell_type].max()
    y_range = y_max - y_min
    
    x1, x2 = 0, 1
    h = y_max + (y_range * 0.12)
    tip = y_range * 0.03

    # Draw the bracket
    ax.plot([x1, x1, x2, x2], [h - tip, h, h, h - tip], lw=1.2, c='black')
    # Add the stars
    ax.text((x1 + x2) * 0.5, h, get_star_label(p_val), 
            ha='center', va='bottom', fontsize=12, fontweight='bold')

    # Clean up Labels
    ax.set_title(f'{cell_type_clean}', fontsize=14, weight='bold')
    ax.set_xlabel('', fontsize=10)
    ax.set_ylabel('Proportion', fontsize=11, weight='bold')
    
    # Spine cleanup for paper look
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

# Main Title and Save
fig.suptitle('Cell Type Proportions between Clusters', fontsize=24, weight='bold', y=1.02)
plt.tight_layout(rect=[0, 0, 1, 0.98])

save_path = f"figures/{model_name}_patientstrat_ct_abundance_boxplots_paperstyle.pdf"
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"Figure saved to: {save_path}")
plt.show()

# spatial signatures

In [ ]:
adata = sc.read_h5ad("mofa/resources/v2_visium_concat_deconvolved_pathways_patho_hoptimus1.h5ad")

In [ ]:
def quantile_normalize(df: pd.DataFrame) -> pd.DataFrame:
    sorted_vals = np.sort(df.values, axis=0)
    rank_means = np.mean(sorted_vals, axis=1)
    ranks = df.rank(method='min', axis=0).astype(int) - 1
    return pd.DataFrame(rank_means[ranks.values], index=df.index, columns=df.columns)


def normalize_visium(adata: sc.AnnData, target_sum: float = 1e6):
    # Library‐size adjust (CPM)
    sc.pp.normalize_total(adata, target_sum=target_sum, inplace=True)

    # Log2(+1)
    sc.pp.log1p(adata)

    # Quantile-normalize across spots
    X = adata.X.toarray() if scipy.sparse.issparse(adata.X) else adata.X
    df = pd.DataFrame(X.T, index=adata.var_names, columns=adata.obs_names)
    df_qn = quantile_normalize(df)
    # transpose back: spots × genes
    X_qn = df_qn.T.values

    # Safe per-gene z-score (mean=0, sd=1; avoid divide-by-zero)
    gene_means = X_qn.mean(axis=0)
    gene_sds = X_qn.std(axis=0, ddof=1)
    gene_sds[gene_sds == 0] = 1
    X_zscored = (X_qn - gene_means) / gene_sds

    adata.X = X_zscored  # now fully normalized

In [ ]:
def calc_gene_signature(sample_name, view, factor_num, adata, ax=None):
    # Load and Process the MOFA Weight Data 
    weights_df = pd.read_csv(f"mofa/mofa_workflow/results_mosaic/03_results/03_Weight_Data_{model_name}.csv")

    weights_momac = weights_df[weights_df['type'] == view].copy()
    # Extract the gene name from the 'variable_name' column 
    weights_momac['gene'] = weights_momac['variable_name'].str.split("__").str[1]

    # Select the desired factor and extract gene weights
    factor_col = f"Factor{factor_num}"
    gene_weight_dict = weights_momac.set_index('gene')[factor_col].to_dict()

    # Load Visium Data 
    sample = adata[adata.obs["sample_id"] == sample_name]

    print(f"Normalizing {sample_name} …")
    normalize_visium(sample)

    # Compute the Weighted Gene Signature Score for Each Spot
    # Build a weight vector aligned with the Visium gene order.
    weights_vector = np.array([gene_weight_dict.get(gene, 0) for gene in sample.var_names])

    # Compute the signature scores using the appropriate dot product
    if scipy.sparse.issparse(sample.X):
        signature_scores = sample.X.dot(weights_vector)
    else:
        signature_scores = np.dot(sample.X, weights_vector)

    # Save the computed scores in the observation annotations
    sample.obs[f'signature_{view}_fac{factor_num}'] = signature_scores

    # Plot the spatial representation on the provided axis
    if ax is not None:
        sc.pl.spatial(sample, 
                        color=f'signature_{view}_fac{factor_num}', 
                        title=sample_name, 
                        ax=ax, 
                        show=False)

    df = sample.obs[[f'signature_{view}_fac{factor_num}']].copy()
    df['barcode'] = sample.obs_names         # add barcodes
    df = df.reset_index(drop=True)[['barcode', f'signature_{view}_fac{factor_num}']]

    return df

In [ ]:
def calc_patho_samples(sample_names, view, factor, adata):

    dfs = []
    # Iterate through each sample with tqdm progress bar
    for i, sample_name in enumerate(tqdm(sample_names, desc="Processing samples")):
        df = calc_gene_signature(sample_name, view, factor_num=factor, adata=adata)
        df['sample'] = sample_name
        dfs.append(df)

    combined_df = pd.concat(dfs, ignore_index=True)
    combined_df.to_csv(f"mofa/spatial_signatures/combined_gene_signatures_{model_name}_fac{factor}_{view}.csv")

In [ ]:
# all at once per factor

In [ ]:
long_samples = []
for i in indices_long:
    sample_id = long[i]
    long_samples.append(sample_id)

medium_samples = []
for i in indices_medium:
    sample_id = medium[i]
    medium_samples.append(sample_id)

short_samples = []
for i in indices_short:
    sample_id = short[i]
    short_samples.append(sample_id)

In [ ]:
all_samples = long_samples + medium_samples + short_samples

In [ ]:
weights_df = pd.read_csv(f"mofa/mofa_workflow/results_mosaic/03_results/03_Weight_Data_{model_name}.csv")
sample_names = all_samples
views = list(weights_df["type"].unique())
factors = ["7"]

In [ ]:
views.remove('celltype_proportions_zComp_CLR') # cant calc signature with genes on that

In [ ]:
for factor in factors:
    for view in views:
        print(f"Calc spatial signatures for fac {factor} and view: {view}...")
        calc_patho_samples(sample_names, view, factor, adata=adata)

In [ ]:
# all at once
dir_path = "mofa/spatial_signatures"

In [ ]:
dfs = []
for csv in os.listdir(dir_path):
    if csv.endswith(".csv"):
        if model_name in csv:
            print(csv)
            df = pd.read_csv(os.path.join(dir_path, csv), index_col=0)
            del df['sample']
            dfs.append(df)

In [ ]:
combined_on_barcode = reduce(
    lambda left, right: pd.merge(left, right, on='barcode', how='outer'),
    dfs
)

In [ ]:
combined_on_barcode

In [ ]:
# aggregate
fac7_cols = [c for c in combined_on_barcode.columns if c.endswith('_fac7')]
combined_on_barcode['signatures_fac7_agg'] = combined_on_barcode[fac7_cols].sum(axis=1)

In [ ]:
combined_on_barcode.set_index('barcode', inplace=True)

In [ ]:
df_obs = combined_on_barcode.reindex(adata.obs_names)

In [ ]:
for col in df_obs.columns:
    adata.obs[col] = df_obs[col]

In [ ]:
adata.write(f"mofa/resources/visium_{model_name}_withfac7_sign_shortmedlong.h5ad")

In [ ]:
fac7_columns = [col for col in adata.obs.columns if 'fac7' in col]

In [ ]:
# Loop through each sample as before
for patho_sample in sample_names:
    if patho_sample in long_samples:
        group = "long"
    elif patho_sample in medium_samples:
        group = "medium"
    elif patho_sample in short_samples:
        group = "short"
    else:
        print(f"Group not found for {patho_sample}.")
        continue
        
    print(f"Plotting for {patho_sample}, {group}")
    sample = adata[adata.obs['sample_id']==patho_sample]

    sc.pl.spatial(
        sample,
        color=fac7_columns,  
        spot_size=150,
        cmap='viridis',
        show=False,
        save=f"_{model_name}_spatialsign_{patho_sample}_{group}_all_fac7.pdf"
    )

In [ ]:
for patho_sample in sample_names:
    print(f"Plotting for {patho_sample}")
    sample = adata[adata.obs['sample_id']==patho_sample]
    sc.pl.spatial(sample, color=['signatures_fac7_agg'], spot_size=150, save=f"{model_name}_spatialsign_reverse_{patho_sample}_fac7", cmap='viridis_r')

# proxy signature and validation with TCGA/CCGA

In [ ]:
data1 = pd.read_csv(f"mofa/resources/patient_strat_{model_name}.csv")
groups = data1[['sample_id', 'Cluster']]

In [ ]:
result_df = groups
meta_path = '/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/data/mosaic_dataset/Clinical/GBM_HK_sample_and_clinical_data.csv'
clinical_cols: list[str] = ['os_years','os_censor','pfs_years','pfs_censor', 'sample_id']
id_col: str = 'sample_id'

In [ ]:
meta = pd.read_csv(meta_path)
# normalize sample_id format
meta[id_col] = meta[id_col].str.replace('_', '')

In [ ]:
merged_meta_mofa = pd.merge(left = meta[clinical_cols], right = groups, on=id_col)

In [ ]:
merged_meta_mofa

In [ ]:
merged_meta_mofa.to_csv(f"mosaic_samples_pfs_os_cluster_{model_name}.csv")

In [ ]:
# selection of genes as input

In [ ]:
res = pd.read_csv(f"mofa/resources/{model_name}deseq2_cluster1_vs_0_results.csv", index_col=0)

In [ ]:
top_hits = (
    res[res["signif"]]
    .sort_values("padj_plot", ascending=True)
)

In [ ]:
genes = top_hits.index.values

In [ ]:
# load and subet bulk from mosaic
data2 = pd.read_csv("mofa/mofa_workflow/input_data_mosaic/bulk_05symbols.csv")
data2.set_index('sample_id', inplace=True)
samples_in_both = data2.index.intersection(merged_meta_mofa["sample_id"])
data2 = data2.loc[samples_in_both, :]

In [ ]:
# read TCGA bulk

In [ ]:
tcga = pd.read_csv("final_expression_with_survival.csv", index_col=0)

In [ ]:
overlap_genes = list(set.intersection(set(genes), set(tcga.columns), set(data2.columns)))

In [ ]:
data2_filtered = data2[overlap_genes]
data2_filtered

In [ ]:
tcga_filtered = tcga[overlap_genes]
tcga_filtered

## regression on factor values

In [ ]:
facs = pd.read_csv(f"mofa/mofa_workflow/results_mosaic/03_results/03_Factor_Data_{model_name}.csv")
groups = facs[['sample_id', 'Factor7']]

In [ ]:
result_df = groups
meta_path = '/mnt/custom-file-systems/efs/fs-09913c1f7db79b6fd/data/mosaic_dataset/Clinical/GBM_HK_sample_and_clinical_data.csv'
clinical_cols: list[str] = ['os_years','os_censor','pfs_years','pfs_censor', 'sample_id']
id_col: str = 'sample_id'

In [ ]:
meta = pd.read_csv(meta_path)
# normalize sample_id format 
meta[id_col] = meta[id_col].str.replace('_', '')

In [ ]:
merged_meta_mofa = pd.merge(left = meta[clinical_cols], right = groups, on=id_col)

In [ ]:
tcga = pd.read_csv("final_expression_with_survival.csv", index_col=0)

In [ ]:
# load and subet bulk from mosaic
data2 = pd.read_csv("mofa/mofa_workflow/input_data_mosaic/bulk_05symbols.csv")
data2.set_index('sample_id', inplace=True)
samples_in_both = data2.index.intersection(merged_meta_mofa["sample_id"])
data2 = data2.loc[samples_in_both, :]

In [ ]:
#overlap_genes = list(set.intersection(set(tcga.columns), set(data2.columns))) # not reproducible. executed once and saved results to load in next line

In [ ]:
order = pd.read_csv("xval_001_order.csv",index_col=0).columns.values # read in the order from first execution for reproducability

In [ ]:
overlap_genes = order

In [ ]:
len(overlap_genes)

In [ ]:
tcga_filtered = tcga[overlap_genes]
tcga_filtered

In [ ]:
data2_filtered = data2[overlap_genes]
data2_filtered

In [ ]:
## Normalize RNA-seq Count Data
def normalize_counts(counts_df):
    """Normalizes a raw count dataframe to log2(CPM+1)."""
    # Calculate library size (total reads per sample)
    library_size = counts_df.sum(axis=1)
    # Calculate Counts Per Million (CPM)
    cpm = counts_df.div(library_size, axis=0) * 1_000_000
    # Return log2(CPM + 1) transformed data
    return np.log2(cpm + 1)

# Apply the normalization to both datasets
data2_filtered_norm = normalize_counts(data2_filtered)
tcga_filtered_norm = normalize_counts(tcga_filtered)


In [ ]:
## Prepare Data for Modeling
# Align mosaic expression data (X) with factor data (y)
X_mosaic = data2_filtered_norm
# Ensure the survival data is a Series with sample_id as the index
y_mosaic = merged_meta_mofa.set_index('sample_id')['Factor7'].loc[X_mosaic.index]

# Align TCGA expression data (X_val) with its survival data (y_val)
X_val = tcga_filtered_norm


## Split the Mosaic Data into Training and Test Sets
X_train, X_test, y_train, y_test = train_test_split(
    X_mosaic,
    y_mosaic,
    test_size=0.2, # 80% for training, 20% for internal testing
    random_state=42 # for reproducibility
)
print(f"\nMosaic data split: {len(X_train)} training samples, {len(X_test)} test samples.")

In [ ]:
## Feature Scaling and Model Training
# Initialize the scaler
scaler = StandardScaler()

# Fit the scaler ONLY on the training data to avoid data leakage
X_train_scaled = scaler.fit_transform(X_train)

# Initialize the ElasticNetCV model.

enet_cv_model = ElasticNetCV(
    cv=5, 
    random_state=42, 
    l1_ratio=[.1, .5, .7, .9, .95, .99, 1], 
    max_iter=10000
)

# Train the model
print("Training ElasticNetCV model to find the best regularization...")
enet_cv_model.fit(X_train_scaled, y_train)

# Print the optimal parameters it found
print(f"\nOptimal alpha found: {enet_cv_model.alpha_:.5f}")
print(f"Optimal L1 ratio found: {enet_cv_model.l1_ratio_:.2f}")

# Check how many features it selected
num_selected_features = np.sum(enet_cv_model.coef_ != 0)
print(f"Number of genes selected by the model: {num_selected_features}")


## Evaluate on the Internal Test Set (Mosaic)
print("\n--- Evaluating on Internal Test Set (Mosaic) ---")
# Use the scaler that was FIT on the training data to transform the test data
X_test_scaled = scaler.transform(X_test)

# Make predictions
y_pred_test = enet_cv_model.predict(X_test_scaled)

# Calculate metrics
mse_test = mean_squared_error(y_test, y_pred_test)
r2_test = r2_score(y_test, y_pred_test)

print(f"R-squared (R²): {r2_test:.4f}")
print(f"Mean Squared Error (MSE): {mse_test:.4f}")
print(f"Root Mean Squared Error (RMSE): {np.sqrt(mse_test):.4f}")

In [ ]:
plt.figure(figsize=(6,6))
sns.scatterplot(x=y_test, y=y_pred_test)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')  # diagonal line
plt.xlabel("True Factor value")
plt.ylabel("Predicted Factor value")
plt.show()

In [ ]:
# --- 1. Load the trained ElasticNetCV model ---
with open('enet_cv_model_001.pkl', 'rb') as f:
    enet_cv_model = pickle.load(f)
    
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

In [ ]:
# Evaluate on the Internal Test Set (Mosaic)
X_test_scaled = scaler.transform(X_test)

# Make predictions
y_pred_test = enet_cv_model.predict(X_test_scaled)

In [ ]:
# Define the threshold (median)
median_threshold = -0.0067658435962317

y_test_cat = np.where(y_test >= median_threshold, 1, 0)
y_pred_cat = np.where(y_pred_test >= median_threshold, 1, 0)

# Define labels for the matrix
# 0 represents 'Low Factor 7', 1 represents 'High Factor 7'
labels = [0, 1]
label_names = ['Low Factor 7', 'High Factor 7']

# 4. Compute the confusion matrix
cm = confusion_matrix(y_test_cat, y_pred_cat, labels=labels)

# 5. Plot the confusion matrix using a heatmap
plt.figure(figsize=(8, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_names, 
            yticklabels=label_names,
            cbar=False)

plt.xlabel('Predicted Factor 7 Category')
plt.ylabel('Actual Factor 7 Category')
plt.title('Confusion Matrix: High vs Low Factor 7')
#plt.show()
plt.savefig("figures/confmatrix_proxysignature.pdf")

In [ ]:
plt.figure(figsize=(6,6))
sns.scatterplot(x=y_test, y=y_pred_test)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')  # diagonal line
plt.xlabel("True Factor 7 value")
plt.ylabel("Predicted Factor 7 value")
#plt.title(f"Test set: R²={r2_score(y_true,y_pred):.2f}, RMSE={mean_squared_error(y_true,y_pred, squared=False):.1f}")
#plt.show()
plt.savefig("figures/regression_proxysignature.pdf")

In [ ]:
## 6. Evaluate on the External Validation Set (TCGA)
print("\n--- Evaluating on External Validation Set (TCGA) ---")
# IMPORTANT: Apply the SAME scaler from the mosaic training data
X_val_scaled = scaler.transform(X_val)

# Make predictions using the trained model
y_pred_val = enet_cv_model.predict(X_val_scaled)

In [ ]:
# save
with open('enet_cv_model_001.pkl','wb') as f:
    pickle.dump(enet_cv_model,f)

In [ ]:
X_val.to_csv("xval_001_order.csv")

In [ ]:
median = -0.0067658435962317

In [ ]:
Cluster_pred = (y_pred_val>median).astype(int) 

In [ ]:
#Cluster_pred

In [ ]:
tcga_os = tcga[["OS_STATUS","OS_TIME"]]
tcga_os["OS_TIME"] = tcga_os["OS_TIME"] / 365
tcga_pfs = tcga[["PFS_STATUS","PFS_TIME"]]
tcga_pfs["PFS_TIME"] = tcga_pfs["PFS_TIME"] / 365

In [ ]:
tcga_df = tcga_pfs
tcga_df = tcga_os

In [ ]:
# Combine TCGA PFS data with our model's predicted clusters
validation_results = tcga_df.loc[X_val.index].copy()
validation_results['predicted_cluster'] = Cluster_pred

In [ ]:
validation_results['predicted_cluster'].value_counts()

In [ ]:
# remove dublicate samples in TCGA (see methods section in the paper)
remove = ['19f6db33-bd74-44aa-9f00-c31903e88abf', 'f34feebc-700f-4cd2-b04d-fa2c8231c56e', 'c402effd-076f-4d5b-a0e1-8b97893aaabd', 'e60d688d-34e7-4624-9897-7163d35b62ed', '9e79eb6c-bdfa-4deb-b283-7c351dac4434', 'f426768e-744a-4bed-bb8e-b5957a350bf2', 'da7d327b-1022-4ffc-9ed8-8b4413209ce4', '210daf0d-280e-4c43-a48b-65e5c2a92a77', 'ce5fb607-99c5-457f-9022-df38b540fce6', 'e925c7f7-d9a2-46a5-9f78-77f9245ff2e1', '8a1f814a-8f47-47e2-870b-8b2aa4df9ccf', 'a6d109ab-c352-463e-a786-ea1ad30f78cf', 'df3438b0-f3c9-4d19-bfe3-685b97b05a3a', 'd02fcfe5-6834-4c66-93d4-768783500ede', '8ed4d0ab-b30d-4775-a533-18fac1a0db86', '6ff649cb-781f-4d71-b06f-3504d95879ad', '77405f24-ebc0-45be-9beb-ac81ed1936d5', 'f54636e5-d823-4bc2-9475-2f4f81879067', '9fefe100-9765-4b8a-8468-51df066f0f0c', '68e36a61-182e-4dd3-ab3e-2d699edcecb3', 'd561fbb1-a288-4a37-98de-65247968fa40', 'fae111e5-9372-4e6d-8e0d-c158c478afc4', '8ce23744-e0bc-459b-8b6d-8bac6c4ef6d5', 'e84acff3-3067-49e4-82b7-dbbddfddb723', '8a3c7c61-631f-4dd0-83fe-d2bbb33a17a4', 'e58cdc72-ac07-40cd-90c5-ebb2d0b8321d', '748dc14a-ff43-4b5d-b197-b7c021955fed', 'ed0f8f0e-9dbf-475b-8cd7-1f712c5929b8', 'fd18cdd5-f7da-45d6-aaa1-d811a4f0a4e1', 'b35f91de-4d52-421d-a36d-05c4cb69a87d', '33ce2e6b-71eb-459d-9b05-9495a21be871', '5862a2c8-9700-4bfd-8247-176efebc3efb', 'f505fb40-a27a-4a6a-9a3d-1f11a4b834b3', 'c6b53ece-179d-4eb7-b4e0-7e5ba2a06a03', '2ee60e6d-094f-4a47-b8e7-fa4f16ec040a', 'c247006d-7749-4838-9bfb-1388e089450d', '7c8860f3-25df-4c30-9d1c-8e9fbc5dc49f', 'a7bb2b43-1c1f-4dff-a696-348f4a4df51f', 'e3ca6a99-2907-4509-a416-eb7362b0b606', '9cee9b1d-e8ee-4f8e-be85-c96dac6138b3', 'a6384a57-0d59-4e92-8ca2-07bd22cf906f', 'f990759a-0fb8-44f5-8ee3-b69d00ea808d', '37253fe8-4a9a-4076-9244-8cc727d6b432', 'd98b999b-c625-4970-9b6a-533e7a4b7a0f', '3a7fd71e-9336-46e0-8106-3df32aed7c20', 'aed30b17-7f0e-4a3c-be14-3b4384b04340', '6afd066b-806e-4568-94d9-ac189d1101af', '5763b7c5-89cb-49c4-87aa-bad01f28b541', '28016f07-a161-43bc-954d-d5ad756bb40b', 'b64598e5-b8fc-4de1-8c6d-7418e867f0d3', 'fa094819-3bc6-4bc6-92cc-906254bbfa73', 'b21a2a30-3ff8-482f-add4-f4eea1204b87', '39f7b501-0578-4c4e-9fc4-fd9a7d839dbf', 'dad61e18-e3f1-4beb-b3c3-ae434e35af2d', 'f60c516c-6be7-4dda-9e46-46a87039c099', 'c70bd18d-2a4e-4a6f-911d-cccf47dac0d6', 'e8c1b868-8139-4831-9c66-32f22d556f29', 'b5a77ad0-9041-4392-afde-ff98f1321c97', '2c20375f-8a54-4f45-a3bf-3cf63c44c678', 'e3c51e59-d9ff-425c-ade2-2a396333826a', '80f616bc-e28d-43b2-b3cb-606962c0d22b', 'ceb8abb5-067a-475c-99b2-128d66267b3f', '5d732c32-681d-46ca-b0bd-404ab47c64db', '7a5f6106-e991-4fdb-ae52-61099bd8c3b9', 'e0497b17-99ef-4d0b-afc9-9fcd5769baaa', '97f42a19-6ae6-4cd5-b4f5-4777c44324e8', '27388014-4a95-4b65-b488-79be5bf9f5b0', 'b44a40d4-3233-4b45-b0b3-799823970345', '7aaf54b8-2be8-4b92-91c9-cfc21671dc60', '917fa61d-4556-42a0-bc3a-449ef1b29886', '0f89ce0f-1176-4c48-af82-4917ccc8f919', 'cdd06204-36e8-4cf9-90c2-f96149f5f58e', '48a76a0c-4e67-466a-aeaa-fe2eafde4c03', 'd7b7ba93-518f-4ee9-a966-d5eb7c3cc8f3', 'd9ce8aca-09dc-4140-a530-e6412c0ad927', '967e6f4b-8c6b-4498-87c2-330b2c174061', 'e1757a20-2d6f-4aee-bafb-804302b448ea', 'a9b55a15-9678-4a04-9041-8e2f5a2b14f1', 'd838610b-badb-407b-b65b-22abf5fc0997', 'efbd20eb-2922-4904-8cf1-88d5bb09ab30', 'cebc70af-a3df-45ef-9656-050f382f64e8', '14141d1c-c65a-4f28-9fb4-079a5950c25c', 'a1de7a70-a829-4491-92ab-8db44391ba19', 'aa132147-3074-472a-859f-f65b484a9c35', 'f489cec9-6f4b-4364-b85a-d34cbc8c2015', '64d208b9-6a15-4077-885f-30d9cdedf147', '8369ba66-e7c0-4340-b7f7-8065342f5c3a', '45ab4192-f1bb-46cd-8e85-c2341d67642f', 'ef9cc9ef-1cf3-46fc-b3cb-06ea0582b7fa', 'c1095377-3814-4ab9-a979-1e4c116c2f22', '9f962dec-eac6-423c-a6ba-22d96165c616', 'fd950816-2bc3-4e25-90b4-6a6a049cd809', '39f300fa-3783-4b58-85fe-5ae8fb8e04b1', 'b18fea3f-d240-4f38-b1c0-5b8cd2ac6d38', 'e044dbf9-7587-48b7-b58a-b4f5d9401309', 'd7f50a7e-b1e9-47ea-b33a-856964b813f7', '88c49e2a-ad65-411f-b9d9-df83de6f2b2a', 'b11f8af1-b2a1-474c-a6d7-e28fc0df6885', '93803595-0b8a-4d0b-9439-e1d7d1eb6fd4', 'b5402a87-05a5-421f-be4c-469c34e4bb81', 'a4e0b059-6e1e-4e90-b55a-4e043e6ed051', 'f78bc420-1d4a-4b75-b400-65352b99b4e8', '9f8c56eb-1505-4eb5-acc5-76609949cecd', 'c1c1d4e2-83de-43de-b586-9b375dc91f4d', '4a91d152-b46f-4721-a868-7ed491f9af56', 'b198a226-f734-46c8-93db-0914f0fa6805', '15721800-d7bf-49ef-91cb-00fea9254137', '7412a444-9fad-4d96-a4d4-2c270045919a', '388193e4-004b-44b4-9eac-267eab7a9412', '9336dd74-2f3f-4c1f-bebf-e4aae364f784']

In [ ]:
validation_results = validation_results.drop(labels=remove)

In [ ]:
validation_results.rename(columns={"OS_STATUS": "os_censor", "OS_TIME": "os_years"}, inplace=True)

In [ ]:
validation_results.to_csv("results_001.csv")

In [ ]:
validation_results = pd.read_csv("results_001.csv", index_col=0)

In [ ]:
# Kaplan-Meier Analysis
kmf = KaplanMeierFitter()
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

# Generate a survival curve for each predicted cluster
for cluster_label in sorted(validation_results['predicted_cluster'].unique()):
    group = validation_results[validation_results['predicted_cluster'] == cluster_label]
    kmf.fit(group['os_years'], 
            event_observed=group['os_censor'], 
            label=f'Predicted Cluster {cluster_label} (n={len(group)})')
    #kmf.plot(ax=ax)
    kmf.plot(ci_show=True, ax=ax, color=COLOR_PALETTE[cluster_label])

# Perform the log-rank test to compare the survival curves
groups = validation_results['predicted_cluster']
ix0 = (groups == 0)
ix1 = (groups == 1)

results = logrank_test(
    durations_A=validation_results.loc[ix0, 'os_years'],
    event_observed_A=validation_results.loc[ix0, 'os_censor'],
    durations_B=validation_results.loc[ix1, 'os_years'],
    event_observed_B=validation_results.loc[ix1, 'os_censor']
)

# Final plot formatting and statistical output
plt.title('Kaplan-Meier Survival Curves by Predicted Cluster (TCGA Cohort)')
plt.xlabel('Overall Survival (Years)')
plt.ylabel('Survival Probability')
plt.grid(False)
# annotate rounded p-value on the plot
plt.text(
    0.18, 0.02,
    f'p = {results.p_value:.5f}',
    ha='right', va='bottom',
    transform=ax.transAxes,
    fontsize=12,
    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.8)
)
#plt.text(0.05, 0.05, f'Log-Rank Test p-value: {results.p_value:.4f}', transform=ax.transAxes, fontsize=12)
plt.legend()
plt.savefig(f"figures/kaplan_meier_os_TCGA_{model_name}.pdf")
plt.show()

### CCGA

In [ ]:
ccga = pd.read_csv("ccga_final_expression_with_survival.csv", index_col=0)

In [ ]:
ccga_filtered = ccga.iloc[:,:-2]

In [ ]:
ccga_filtered_norm = normalize_counts(ccga_filtered)

In [ ]:
X2_val = ccga_filtered_norm

In [ ]:
X2_val_scaled = scaler.transform(X2_val)

# Make predictions using the trained model
y2_pred_val = enet_cv_model.predict(X2_val_scaled)

In [ ]:
Cluster_pred2 = (y2_pred_val>median).astype(int)

In [ ]:
ccga_os = ccga[["OS_STATUS","OS_TIME"]]

In [ ]:
ccga_df = ccga_os

In [ ]:
# Combine TCGA PFS data with model's predicted clusters
validation_results2 = ccga_df.loc[X2_val.index].copy()
validation_results2['predicted_cluster'] = Cluster_pred2

In [ ]:
validation_results2.rename(columns={"OS_STATUS": "os_censor", "OS_TIME": "os_years"}, inplace=True)

In [ ]:
validation_results2 = validation_results2.dropna(subset=['os_years']) 

In [ ]:
validation_results2 = pd.read_csv("results_ccga_003.csv",index_col=0) # saved later

In [ ]:
# Kaplan-Meier Analysis
kmf = KaplanMeierFitter()
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

# Generate a survival curve for each predicted cluster
for cluster_label in sorted(validation_results2['predicted_cluster'].unique()):
    group = validation_results2[validation_results2['predicted_cluster'] == cluster_label]
    kmf.fit(group['os_years'], 
            event_observed=group['os_censor'], 
            label=f'Predicted Cluster {cluster_label} (n={len(group)})')
    #kmf.plot(ax=ax)
    kmf.plot(ci_show=True, ax=ax, color=COLOR_PALETTE[cluster_label])


# Perform the log-rank test to compare the survival curves
groups = validation_results2['predicted_cluster']
ix0 = (groups == 0)
ix1 = (groups == 1)

results = logrank_test(
    durations_A=validation_results2.loc[ix0, 'os_years'],
    event_observed_A=validation_results2.loc[ix0, 'os_censor'],
    durations_B=validation_results2.loc[ix1, 'os_years'],
    event_observed_B=validation_results2.loc[ix1, 'os_censor']
)

# Final plot formatting and statistical output
plt.title('Kaplan-Meier Survival Curves by Predicted Cluster (CCGA Cohort)')
plt.xlabel('Overall Survival (Years)')
plt.ylabel('Survival Probability')
plt.grid(False)
# annotate rounded p-value on the plot
plt.text(
    0.18, 0.02,
    f'p = {results.p_value:.5f}',
    ha='right', va='bottom',
    transform=ax.transAxes,
    fontsize=12,
    bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='gray', alpha=0.8)
)
#plt.text(0.05, 0.05, f'Log-Rank Test p-value: {results.p_value:.4f}', transform=ax.transAxes, fontsize=12)
plt.legend()
plt.savefig(f"figures/kaplan_meier_os_CCGA_{model_name}.pdf")
plt.show()

In [ ]:
validation_results2.to_csv("results_ccga_003.csv")

In [ ]:
# Save
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

## check features from model

In [ ]:
# Load the trained ElasticNetCV model
with open('enet_cv_model_001.pkl', 'rb') as f:
    enet_cv_model = pickle.load(f)

# Load the X_val file to get feature names
X_val = pd.read_csv('xval_001_order.csv', index_col=0)  # skip the first unnamed index column if present
feature_names = X_val.columns

# Extract coefficients and find selected (non-zero) ones
coefs = enet_cv_model.coef_
selected_mask = coefs != 0

selected_features = feature_names[selected_mask]
selected_weights = coefs[selected_mask]

# Combine into a dataframe
weights_df = pd.DataFrame({
    'feature': selected_features,
    'weight': selected_weights
}).sort_values(by='weight', ascending=False)

# Inspect results
print(f"Number of selected features: {len(weights_df)}")

In [ ]:
weights_df.reset_index(inplace=True)

In [ ]:
weights_df

In [ ]:
weights_df.drop(columns=["index"], inplace=True)

In [ ]:
weights_df.to_csv("enet_cv_model_001_weights.csv")